# Chapters 6–9 — Executable multimodal feature extraction

**[implementation]** This notebook uses PyTorch + timm to instantiate Xception and ResNeXt-50, optionally loading ImageNet weights. It does **not** recover the original research framework, original frozen layer boundaries, or reproduce previous metrics. First run with `pretrained=False` to validate architecture and shapes without downloading weights; then switch to `True` for the intended ImageNet transfer-learning workflow.

In [ ]:
%pip install torch timm

import torch
import timm
from autonomous_driving.backbone_config import get_backbone_assignment
from autonomous_driving.backbone_extraction import (
    BackboneModelNames, available_model_names, make_multimodal_backbones,
    freeze_all_except_last_n_children, flattened_concatenated_features,
)
print('torch:', torch.__version__, 'timm:', timm.__version__)
print('Xception candidates:', available_model_names('*xception*'))
print('ResNeXt candidates:', available_model_names('*resnext50*'))

## Choose an assignment and pretrained policy

A = camera ResNeXt-50, LiDAR Xception. B = camera Xception, LiDAR ResNeXt-50. Model registry names depend on the installed timm release; inspect the printed candidates and update names explicitly if required. Downloaded weights require internet access and compatible licenses.

In [ ]:
CONFIGURATION = 'A'  # Switch to 'B' for the second research assignment
PRETRAINED = False  # Change to True for actual ImageNet-initialized experiments
names = BackboneModelNames()
assignment = get_backbone_assignment(CONFIGURATION)
print(assignment)
camera_backbone, lidar_backbone = make_multimodal_backbones(
    CONFIGURATION, pretrained=PRETRAINED, names=names
)
print('Camera backbone:', type(camera_backbone).__name__)
print('LiDAR backbone:', type(lidar_backbone).__name__)

## Freeze early stages; fine-tune upper stages

**[implementation]** Unfreezing the last direct child module is a demonstration, **not** a recovered research setting. Record the actual module names and trainable parameter counts before running a real experiment.

In [ ]:
camera_freeze = freeze_all_except_last_n_children(camera_backbone, n=1)
lidar_freeze = freeze_all_except_last_n_children(lidar_backbone, n=1)
print('Camera freeze policy:', camera_freeze)
print('LiDAR freeze policy:', lidar_freeze)
for label, model in [('camera', camera_backbone), ('lidar', lidar_backbone)]:
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f'{label}: total={total:,}, trainable={trainable:,}')

## Synthetic shape check: Flatten → Concatenate

This uses zeros for **shape testing only**, not for scientific performance measurement. Pretrained model normalization and real sensor alignment must be implemented separately. CPU inference may take time.

In [ ]:
camera_backbone.eval()
lidar_backbone.eval()
synthetic_camera = torch.zeros(1, 3, 299, 299)
synthetic_lidar = torch.zeros(1, 3, 299, 299)
with torch.no_grad():
    fused = flattened_concatenated_features(
        camera_backbone, lidar_backbone, synthetic_camera, synthetic_lidar
    )
print('Flatten + Concatenate shape:', tuple(fused.shape))
assert fused.ndim == 2 and fused.shape[0] == 1

## What is not yet implemented

- previous image normalization and full LiDAR rasterization
- DBNet loading, camera/LiDAR synchronization, temporal windows
- LSTM → Attention → Dense and dual steering/speed outputs
- Training and validation on real data

**Important:** Flattening two high-resolution spatial feature maps can produce a large LSTM input. Measure the actual fused dimension and memory requirements before allocating temporal batches. The notebook has no saved execution outputs; run it in a controlled environment and archive the results.